In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
# 2) Imports & Paths (재시작 후 여기서부터 실행)
import os, glob, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler

# ✅ 경로만 수정 (Colab 기본 업로드 경로는 /content)
DATA_DIR = "dataset_new/"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
SAMPLE_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
TEST_GLOB = os.path.join(DATA_DIR, "TEST_*.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, " NumPy:", np.__version__)

Device: cuda  NumPy: 1.26.4


In [5]:
# 3) Load train & basic checks
req_cols = {"date","store","menu","store_menu","sales"}
train = pd.read_csv(TRAIN_PATH)
missing = req_cols - set(train.columns)
assert not missing, f"train.csv missing columns: {missing}"
train['date'] = pd.to_datetime(train['date'])
train = train.sort_values(['store_menu','date']).copy()
print("train shape:", train.shape, "unique store_menu:", train['store_menu'].nunique())
train.head(2)

train shape: (102676, 26) unique store_menu: 193


,date_ordinal,date,store,menu,store_menu,menu_cluster,menu_cluster_label,pattern_group,pattern_group_label,sales,...,sin_doy,cos_doy,sin_dow,cos_dow,is_weekend,is_holiday,day_type,_off_run_id,off_run_pos,is_peak_data
0,738521,2023-01-01,느티나무 셀프BBQ,1인 수저세트,느티나무 셀프BBQ_1인 수저세트,1,Weekday-Steady,0,Seasonal-Peak,0,...,0.017202,0.999852,-0.781832,0.62349,1,1,2,1,1,0
216,738522,2023-01-02,느티나무 셀프BBQ,1인 수저세트,느티나무 셀프BBQ_1인 수저세트,1,Weekday-Steady,0,Seasonal-Peak,0,...,0.034398,0.999408,0.000000,1.00000,0,0,1,0,0,0


In [6]:
# 4) Prepare known-future covariates
known_cols_all = [
    'date_ordinal','dow','month','quarter','is_month_start','is_month_end','season',
    'sin_doy','cos_doy','sin_dow','cos_dow','is_weekend','is_holiday','day_type',
    '_off_run_id','off_run_pos','is_peak_data'
]
known_cols = [c for c in known_cols_all if c in train.columns]
static_cats = [c for c in ["store","menu","menu_cluster_label","pattern_group_label"] if c in train.columns]
for c in static_cats:
    if not pd.api.types.is_numeric_dtype(train[c]):
        train[c] = train[c].astype('category').cat.codes
known_cols += static_cats
print('Known cols used:', known_cols)

Known cols used: ['date_ordinal', 'dow', 'month', 'quarter', 'is_month_start', 'is_month_end', 'season', 'sin_doy', 'cos_doy', 'sin_dow', 'cos_dow', 'is_weekend', 'is_holiday', 'day_type', '_off_run_id', 'off_run_pos', 'is_peak_data', 'store', 'menu', 'menu_cluster_label', 'pattern_group_label']


In [7]:
# 5) Build global series lists
target_series, future_covs = [], []
for key, g in train.groupby('store_menu'):
    g = g.sort_values('date').set_index('date') # Set date as index
    ts = TimeSeries.from_times_and_values(g.index, g['sales'].astype(float), fill_missing_dates=True, freq='D')
    fc = TimeSeries.from_times_and_values(g.index, g[known_cols].astype(float).values, columns=known_cols,
                                          fill_missing_dates=True, freq='D')
    target_series.append(ts)
    future_covs.append(fc)

scaler_y = Scaler()
target_series_scaled = scaler_y.fit_transform(target_series)
scaler_cov = Scaler()
future_covs_scaled = scaler_cov.fit_transform(future_covs)
len(target_series_scaled), len(future_covs_scaled)

(193, 193)

In [8]:
# 6) Train TFT (input=28, output=7)
model = TFTModel(
    input_chunk_length=28,
    output_chunk_length=7,
    hidden_size=64,
    lstm_layers=1,
    dropout=0.2,
    add_relative_index=True,
    batch_size=256,
    n_epochs=25,
    random_state=42,
    # torch_device=DEVICE, # Removed as per error
)
model.fit(series=target_series_scaled, future_covariates=future_covs_scaled, verbose=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0      | train
1  | val_metrics                       | MetricCollection                 | 0      | train
2  | input_embeddings                  | _MultiEmbedding                  | 0      | train
3  | static_cova

Epoch 24: 100%|██████████| 376/376 [01:12<00:00,  5.19it/s, train_loss=0.283]

`Trainer.fit` stopped: `max_epochs=25` reached.


Epoch 24: 100%|██████████| 376/376 [01:12<00:00,  5.19it/s, train_loss=0.283]


TFTModel(output_chunk_shift=0, hidden_size=64, lstm_layers=1, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.2, hidden_continuous_size=8, categorical_embedding_sizes=None, add_relative_index=True, loss_fn=None, likelihood=None, norm_type=LayerNorm, use_static_covariates=True, input_chunk_length=28, output_chunk_length=7, batch_size=256, n_epochs=25, random_state=42)

In [9]:
# 7) Prediction utils — STRICT TEST 28 + 7 only (batch version)
def postprocess(arr):
    arr = np.where(arr < 0, 0, arr)
    mask = (arr > 0) & (arr < 1)
    arr[mask] = 1.0
    return arr

def _ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns and not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def make_series_from_test(test_df, known_cols):
    """각 TEST 파일에서 유효한 시리즈만 모아 '리스트'로 반환 (경고/무한루프 방지)"""
    test_df = test_df.copy()
    test_df['date'] = pd.to_datetime(test_df['date'])

    # static categoricals → 코드화
    static_cats = [c for c in ["store","menu","menu_cluster_label","pattern_group_label"] if c in test_df.columns]
    for c in static_cats:
        if not pd.api.types.is_numeric_dtype(test_df[c]):
            test_df[c] = test_df[c].astype('category').cat.codes

    test_df = _ensure_numeric(test_df, known_cols + ['sales'])

    keys, series_list, cov_list, last_obs_dates = [], [], [], []

    for key, g in test_df.groupby('store_menu'):
        g = g.sort_values('date').set_index('date')

        obs_part = g[g['sales'].notna()].tail(28)
        fut_part = g[g['sales'].isna()].head(7)
        if len(obs_part) != 28 or len(fut_part) != 7:
            continue

        # 35일 cov 인덱스
        cov_index = pd.date_range(start=obs_part.index[0],
                                  end=obs_part.index[-1] + pd.Timedelta(days=7),
                                  freq='D')
        cov_block = g.reindex(cov_index)[known_cols]
        cov_block = cov_block.fillna(method='ffill').fillna(0)
        if len(cov_block) != 35 or cov_block.isna().any().any():
            continue

        ts = TimeSeries.from_times_and_values(
            obs_part.index, obs_part['sales'].astype(float),
            fill_missing_dates=True, freq='D'
        )
        fc = TimeSeries.from_times_and_values(
            cov_block.index, cov_block.astype(float).values,
            columns=known_cols, fill_missing_dates=True, freq='D'
        )
        if len(ts) != 28 or len(fc) != 35:
            continue

        keys.append(key)
        series_list.append(ts)
        cov_list.append(fc)
        last_obs_dates.append(ts.time_index[-1])

    return keys, series_list, cov_list, last_obs_dates

def predict_test_file_batch(test_path, scaler_y, scaler_cov, model, known_cols):
    """시리즈 리스트 단위로 스케일/예측 → 경고 제거 & 빠름"""
    tdf = pd.read_csv(test_path)
    keys, s_list, c_list, last_obs = make_series_from_test(tdf, known_cols)
    if not s_list:
        return pd.DataFrame(columns=["date","store_menu","pred"])

    s_scaled = scaler_y.transform(s_list)       # 리스트 단위 변환
    c_scaled = scaler_cov.transform(c_list)     # 리스트 단위 변환

    yhat_list = model.predict(
        n=7, series=s_scaled, future_covariates=c_scaled,
        n_jobs=-1, verbose=False
    )

    rows = []
    for key, yhat_s, last_d in zip(keys, yhat_list, last_obs):
        yhat = scaler_y.inverse_transform(yhat_s)
        vals = postprocess(yhat.values().flatten())
        pred_dates = pd.date_range(start=last_d + pd.Timedelta(days=1), periods=7, freq='D')
        for d, v in zip(pred_dates, vals):
            rows.append({"date": pd.Timestamp(d), "store_menu": key, "pred": float(v)})
    return pd.DataFrame(rows)



In [10]:
# 8) Run TEST_00~09 → submission.csv (batch version)
sample = pd.read_csv(SAMPLE_PATH)
test_paths = sorted(glob.glob(TEST_GLOB))
assert len(test_paths) > 0, f"No TEST_*.csv found in {DATA_DIR}"

all_pred = []
for p in test_paths:
    tag = os.path.splitext(os.path.basename(p))[0]
    dfp = predict_test_file_batch(p, scaler_y, scaler_cov, model, known_cols)

    if dfp.empty:
        print(f"[WARN] No valid predictions for {tag} (check lengths & NaNs)")
        continue

    # 정확히 7개 날짜만 태깅
    dates_sorted = sorted(dfp['date'].unique())
    if len(dates_sorted) != 7:
        print(f"[WARN] {tag}: unique pred dates != 7 (got {len(dates_sorted)})")
        # 그래도 진행은 함

    date_to_tag = {d: f"{tag}+{i+1}일" for i, d in enumerate(dates_sorted[:7])}
    dfp['tag'] = dfp['date'].map(date_to_tag)
    all_pred.append(dfp)

assert len(all_pred) > 0, "No predictions generated. Check warnings above."
all_pred = pd.concat(all_pred, ignore_index=True)

out = sample.copy()
store_menu_cols = [c for c in out.columns if c != '영업일자']

# 누락 열 0 채움 안내
missing_cols = [c for c in store_menu_cols if c not in all_pred['store_menu'].unique()]
if missing_cols:
    print(f"[WARN] Missing store_menu in predictions (filled with 0): {len(missing_cols)} items")

piv = all_pred.pivot(index='tag', columns='store_menu', values='pred')
piv = piv.reindex(columns=store_menu_cols)
piv = piv.reindex(index=out['영업일자']).fillna(0)

out.loc[:, store_menu_cols] = piv.values
OUT_PATH = os.path.join(DATA_DIR, 'submission.csv')
out.to_csv(OUT_PATH, index=False)
print('Saved:', OUT_PATH)
out.head(3)


Only 187 TimeSeries (lists) were provided which is lower than the number of series (n=193) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.
Only 187 TimeSeries (lists) were provided which is lower than the number of series (n=193) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=193) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=193) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.
Only 1 TimeSeries (lists) were provided which is lower than the number of 

[WARN] Missing store_menu in predictions (filled with 0): 6 items
Saved: dataset_new/submission.csv


,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,23.908515,0.0,8.256043,25.054661,0.000000,3.786594,5.698360,0.000000,3.054906,...,10.881888,3.758877,1.000000,3.927373,38.214217,4.923167,2.471159,5.582465,1.669311,4.180826
1,TEST_00+2일,3.552316,0.0,1.000000,16.962607,5.785627,5.121662,12.856274,11.276624,0.000000,...,1.000000,1.000000,0.000000,1.000000,12.918160,1.000000,1.000000,5.972921,0.000000,4.032206
2,TEST_00+3일,1.000000,1.0,6.523262,4.641809,11.342228,2.476176,5.653948,0.000000,1.000000,...,12.021865,1.000000,6.474574,1.688400,11.211665,5.104815,1.000000,1.861602,1.819613,3.943452


In [11]:
import pandas as pd

# Load CSV
df = pd.read_csv("dataset_new/submission.csv")

# Replace values between 0 and 1 with 1 (excluding first row and column)
df.iloc[1:, 1:] = df.iloc[1:, 1:].applymap(lambda x: 1 if 0 <= x <= 1 else x)

# Save back to CSV
df.to_csv("dataset_new/submission_updated.csv", index=False)
print("Saved to dataset_new/submission_updated.csv")


Saved to dataset_new/submission_updated.csv
